In [1]:
import os
import pandas as pd
from datetime import datetime 
import duckdb
import unicodedata
import sys
from pathlib import Path
from kedro.framework.startup import bootstrap_project
from kedro.framework.session import KedroSession

# 1. Move to project root if we are in the notebooks folder
if Path.cwd().name == "notebooks":
    os.chdir("..")

# 2. Initialize Kedro
project_path = Path.cwd()
bootstrap_project(project_path)

# 3. Create session and get catalog
session = KedroSession.create(project_path)
context = session.load_context()
catalog = context.catalog

print(f"✅ Kedro context loaded! Project root: {project_path}")

[06/23/26 11:08:23] INFO     Using                                                                  ]8;id=110547;file://c:\Users\User\miniconda3\envs\unis\Lib\site-packages\kedro\framework\project\__init__.py\__init__.py]8;;\:]8;id=717764;file://c:\Users\User\miniconda3\envs\unis\Lib\site-packages\kedro\framework\project\__init__.py#269\269]8;;\
                             'c:\Users\User\miniconda3\envs\unis\Lib\site-packages\kedro\framework\                
                             project\rich_logging.yml' as logging configuration.                                   

[06/23/26 11:08:24] WARNING  c:\Users\User\miniconda3\envs\unis\Lib\site-packages\requests\__init__ ]8;id=107076;file://c:\Users\User\miniconda3\envs\unis\Lib\warnings.py\warnings.py]8;;\:]8;id=944724;file://c:\Users\User\miniconda3\envs\unis\Lib\warnings.py#110\110]8;;\
                             .py:113: RequestsDependencyWarning: urllib3 (2.6.1) or chardet                        
                             (6.0.0.post1)/charset_normalizer (3.4.4) doesn't match a supported                    
                             version!                                                                              
                               warnings.warn(                                                                      
                                                                                                                   

[06/23/26 11:08:25] INFO     Kedro is sending anonymous usage data with the sole purpose of improving ]8;id=620532;file://c:\Users\User\miniconda3\envs\unis\Lib\site-packages\kedro_telemetry\plugin.py\plugin.py]8;;\:]8;id=694845;file://c:\Users\User\miniconda3\envs\unis\Lib\site-packages\kedro_telemetry\plugin.py#242\242]8;;\
                             the product. No personal data or IP addresses are stored on our side. To              
                             opt out, set the `KEDRO_DISABLE_TELEMETRY` or `DO_NOT_TRACK` environment              
                             variables, or create a `.telemetry` file in the current working                       
                             directory with the contents `consent: false`. To hide this message,                   
                             explicitly grant or deny consent. Read more at                                        
                             https://docs.kedro.org/en/stable/about/telemetry/                                     

✅ Kedro context loaded! Project root: g:\Unidades compartidas\Alianzas\3. Data\UNIS\unis-perm-flow


In [2]:
# Add the 'src' directory to the path
sys.path.append(os.path.abspath("src"))
import unis_perm_flow.pipelines.data_processing.nodes as nodes_dproc
import unis_perm_flow.pipelines.calac_activos_baj_grad.nodes as nodes_abg

# Importación de fuentes

In [3]:
unis_calaca_ext  = catalog.load('unis_calendario_extendido')
unis_calaca_uptoday  = catalog.load('unis_calendario_extendido_uptoday')
unis_estaca_sd = catalog.load('unis_estaca_sd')

[06/23/26 11:08:26] INFO     Loading data from unis_calendario_extendido (ParquetDataset)...   ]8;id=739276;file://c:\Users\User\miniconda3\envs\unis\Lib\site-packages\kedro\io\data_catalog.py\data_catalog.py]8;;\:]8;id=439464;file://c:\Users\User\miniconda3\envs\unis\Lib\site-packages\kedro\io\data_catalog.py#1048\1048]8;;\

[06/23/26 11:08:30] INFO     Loading data from unis_calendario_extendido_uptoday               ]8;id=825509;file://c:\Users\User\miniconda3\envs\unis\Lib\site-packages\kedro\io\data_catalog.py\data_catalog.py]8;;\:]8;id=816956;file://c:\Users\User\miniconda3\envs\unis\Lib\site-packages\kedro\io\data_catalog.py#1048\1048]8;;\
                             (ParquetDataset)...                                                                   

                    INFO     Loading data from unis_estaca_sd (ParquetDataset)...              ]8;id=507515;file://c:\Users\User\miniconda3\envs\unis\Lib\site-packages\kedro\io\data_catalog.py\data_catalog.py]8;;\:]8;id=374096;file://c:\Users\User\miniconda3\envs\unis\Lib\site-packages\kedro\io\data_catalog.py#1048\1048]8;;\

In [4]:
unis_estados_calac = catalog.load('unis_estados_calac')

                    INFO     Loading data from unis_estados_calac (ExcelDataset)...            ]8;id=606746;file://c:\Users\User\miniconda3\envs\unis\Lib\site-packages\kedro\io\data_catalog.py\data_catalog.py]8;;\:]8;id=241817;file://c:\Users\User\miniconda3\envs\unis\Lib\site-packages\kedro\io\data_catalog.py#1048\1048]8;;\

# Importación de parámetros

In [6]:
parameters = catalog.load("parameters")

[06/23/26 11:08:32] INFO     Loading data from parameters (MemoryDataset)...                   ]8;id=570139;file://c:\Users\User\miniconda3\envs\unis\Lib\site-packages\kedro\io\data_catalog.py\data_catalog.py]8;;\:]8;id=702974;file://c:\Users\User\miniconda3\envs\unis\Lib\site-packages\kedro\io\data_catalog.py#1048\1048]8;;\

In [7]:
# bajas_calac
bajas_calac = parameters['bajas_calac']
graduados_calac = parameters['graduados_calac']

# Bajas

In [8]:
bajas_calendario_academico = nodes_abg.momento_baja(
    # 1. El DataFrame (Objeto en memoria)
    unis_estaca_sd=unis_estaca_sd, 
    dict_duracion=parameters['graduados_calac']['dict_niveles_duracion'],
    fallback_weeks=parameters['graduados_calac']['graduation_fallback_weeks'],
    
    # 2. Valores extraídos de diccionarios de configuración
    unis_col_fechadef=bajas_calac['unis_col_fechadef'],
    unis_col_fechatemp=parameters['bajas_calac']['unis_col_fechatemp'],
    
    # 3. Nombre del dataset según el catálogo
    unis_calaca= unis_calaca_uptoday.drop(columns= 'cohorte'),
    
    # 4. Parámetros de transformación y cruce
    left_on=parameters['bajas_calac']['merge_left_on'],
    right_on=parameters['bajas_calac']['merge_right_on'],
    group_key=parameters['bajas_calac']['unis_calaca_col_cohorte_inicial'],
    sort_cols=parameters['bajas_calac']['unis_calaca_col_sort']
)

In [9]:
# 1. Obtenemos el Top 10 (mismo proceso que antes)
top_diez_desercion = (
    bajas_calendario_academico
    .loc[:, ['periodo_inicial','nivel', 'fecha_ingreso', 'semana_acumulada', 'di']]
    .groupby(['periodo_inicial','nivel',  'fecha_ingreso', 'semana_acumulada'])
    .agg({'di': 'sum'})
    .reset_index()
    .sort_values(by='di', ascending=False)
    .head(10)
)

# 2. Aplicamos las barras de datos de color rojo
top_diez_estilado = top_diez_desercion.style.bar(
    subset=['di'], 
    color='#f8766d', # Un rojo suave tipo "soft red"
    align='left',    # Las barras nacen desde la izquierda
    vmin=0           # El mínimo es cero para que la escala sea real
)

# Visualizar en el Notebook
top_diez_estilado

,periodo_inicial,nivel,fecha_ingreso,semana_acumulada,di
49,9251.000000,maestria,2025-01-14 00:00:00,23.000000,61
74,9252.000000,maestria,2025-05-13 00:00:00,28.000000,33
30,9243.000000,maestria,2024-08-27 00:00:00,64.000000,29
89,9253.000000,maestria,2025-09-02 00:00:00,11.000000,25
136,9265.000000,maestria,2026-03-10 00:00:00,10.000000,19
118,9261.000000,maestria,2026-01-13 00:00:00,10.000000,17
33,9251.000000,maestria,2025-01-14 00:00:00,3.000000,17
93,9253.000000,maestria,2025-09-02 00:00:00,16.000000,16
73,9252.000000,maestria,2025-05-13 00:00:00,27.000000,15
26,9243.000000,maestria,2024-08-27 00:00:00,39.000000,13


# Graduados

In [10]:
graduados_calendario_academico = nodes_abg.momento_grado(
    # 1. Referencias a datasets/DataFrames
    unis_estaca=unis_estaca_sd,
    unis_calaca=unis_calaca_uptoday.drop(columns= 'cohorte'),
    
    # 2. Configuración de lógica de negocio (Duraciones y Grado Inmediato)
   
    col_gi=parameters['graduados_calac']['graduation_col_gi'],
    
    # 3. Parámetro de seguridad (Semanas por defecto si falla el cálculo)
    dict_duracion=parameters['graduados_calac']['dict_niveles_duracion'],
    fallback_weeks=parameters['graduados_calac']['graduation_fallback_weeks'],
    
    # 4. Llaves para el cruce de tablas (Joins)
    join_left=parameters['graduados_calac']['graduation_join_keys_left'],
    join_right=parameters['graduados_calac']['graduation_join_keys_right']
)

In [11]:
# 1. Obtenemos el Top 10 (mismo proceso que antes)
top_diez_graduacion = (
    graduados_calendario_academico
    .loc[:, ['periodo_inicial','nivel', 'fecha_ingreso', 'semana_acumulada', 'gi']]
    .groupby(['periodo_inicial','nivel',  'fecha_ingreso', 'semana_acumulada'])
    .agg({'gi': 'sum'})
    .reset_index()
    .sort_values(by='gi', ascending=False)
    .head(100)
)

# 2. Aplicamos las barras de datos de color rojo
top_diez_estilado = top_diez_graduacion.style.bar(
    subset=['gi'], 
    color="#41747A", # Un verde suave tipo "soft green"
    align='left',    # Las barras nacen desde la izquierda
    vmin=0           # El mínimo es cero para que la escala sea real
)

# Visualizar en el Notebook
top_diez_estilado

,periodo_inicial,nivel,fecha_ingreso,semana_acumulada,gi
1,9251,maestria,2025-01-14 00:00:00,64,188
0,9243,maestria,2024-08-27 00:00:00,64,80


# Activos

In [12]:
activos_calendario_academico = nodes_abg.momento_activos(
   unis_estaca=unis_estaca_sd,
    unis_calaca=unis_calaca_uptoday.drop(columns= 'cohorte'),
    
    # 2. Configuración de lógica de negocio (Duraciones y Grado Inmediato)
    dict_duracion=parameters['graduados_calac']['dict_niveles_duracion'],
    col_di='di',
    col_gi='gi',
    
    # 3. Parámetro de seguridad (Semanas por defecto si falla el cálculo)
    fallback_weeks=None,
    
    # 4. Llaves para el cruce de tablas (Joins)
    join_left='fecha_activo',
    join_right='fecha_inicio',
    group_key = 'fecha_ingreso'
) 

In [13]:
activos_calendario_academico

,id_inconcert,identificacion,correo_2,nombres,fecha_registro,cohorte,fecha_ingreso,usuario_institucional,nivel,programa,...,semana,semana_acumulada,month,mes_academico,anio_gregoriano,mes_gregoriano,student_journey,tipo,engi,ai
0,4734.0,2.154383e+12,2154382640108@unis edu gt,flor anally morales,2026-05-04,2024-08-27,2024-08-27,famorales@unis.edu.gt,maestria,marketing digital,...,8,64,16,m16,2025,12,qa,ciclo academico 4,1,0
1,41207.0,1.707707e+12,1707707451301@unis edu gt,jose arnulfo vasquez rivas,2026-05-04,2026-03-10,2026-03-10,jvasquezr@unis.edu.gt,maestria,docencia universitaria,...,6,14,4,m4,2026,6,q1,ingreso 1,0,1
2,44879.0,1.709469e+12,1709468940101@unis edu gt,adan daniel moran dias,2026-05-04,2026-03-10,2026-03-10,admoran@unis.edu.gt,maestria,docencia universitaria,...,6,14,4,m4,2026,6,q1,ingreso 1,0,1
3,12941.0,1.733052e+12,1733051801413@unis edu gt,luis alfonso xiloj perez,2026-05-04,2026-03-10,2026-03-10,lxiloj@unis.edu.gt,maestria,docencia universitaria,...,6,14,4,m4,2026,6,q1,ingreso 1,0,1
4,44002.0,1.899472e+12,1899472170101@unis edu gt,helga maria sarti segura de barahona,2026-05-04,2026-03-10,2026-03-10,hsarti@unis.edu.gt,maestria,docencia universitaria,...,6,14,4,m4,2026,6,q1,ingreso 1,0,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
911,25664.0,3.469454e+12,3469453990101@unis edu gt,katherine lee alvarez orozco,2026-05-04,2025-10-28,2025-10-28,klalvarez@unis.edu.gt,maestria,docencia universitaria,...,6,30,8,m8,2026,6,qa,ciclo academico 2,0,1
912,35631.0,3.926166e+12,3926165720101@unis edu gt,patricio luis astolfi,2026-05-04,2025-10-28,2025-10-28,pastolfi@unis.edu.gt,maestria,docencia universitaria,...,6,30,8,m8,2026,6,qa,ciclo academico 2,0,1
913,35259.0,1.613967e+12,1613966820101@unis edu gt,joana eunice alonzo caceres,2026-05-04,2025-10-28,2025-10-28,jealonzo@unis.edu.gt,maestria,marketing digital,...,6,30,8,m8,2026,6,qa,ciclo academico 2,0,1
914,4646.0,2.249718e+12,2249717510101@unis edu gt,carmen gabriela reyes mejia de martinez,2026-05-04,2025-10-28,2025-10-28,cgreyes@unis.edu.gt,maestria,marketing digital,...,6,30,8,m8,2026,6,qa,ciclo academico 2,0,1


In [14]:
unis_estaca_sd['estado'].value_counts()


estado
activo                  890
baja definitiva         643
egresado no graduado    270
activo sin materias      23
Name: count, dtype: int64

# Insumo Modelo de  sobbrevivenncia


In [15]:
columns_survival = ['identificacion', 
                    'id_inconcert', 
                    'periodo_inicial', 
                    'nivel', 
                    'programa', 
                    'semana_acumulada',
                    'month',
                    'mes_gregoriano' ,
                    #'ai', 
                    'di']
                    #'gi', 
                    #'engi', 
                    #'ci']

In [ ]:
unis_estados_calac = catalog.load("unis_estados_calac")
unis_estados_calac_survival = unis_estados_calac.loc[:,columns_survival]

[06/22/26 15:02:30] INFO     Loading data from unis_estados_calac (ExcelDataset)...            ]8;id=838414;file://c:\Users\User\miniconda3\envs\unis\Lib\site-packages\kedro\io\data_catalog.py\data_catalog.py]8;;\:]8;id=381343;file://c:\Users\User\miniconda3\envs\unis\Lib\site-packages\kedro\io\data_catalog.py#1048\1048]8;;\

In [ ]:
unis_estados_calac_survival.to_parquet("data/03_primary/unis_estaca_survival.parquet")